# Phase 4 — Pandas Deep Dive

Pandas is the core data manipulation library in Python data science. Nearly every real-world DS project starts here.

**Coverage:** DataFrame/Series, reading data, selection, filtering, groupby, merge/join, reshaping, missing data, apply.

---

**use case of pandas**
* Data Cleaning
* Data Analysis
* Data Transformation
* Data Visualization (Basic Level)
* Data Aggregation
* File Handling
* Data Filtering & Selection
* Time Series Analysis

In [12]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

np.random.seed(42)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 100)

---
## 1. DataFrame & Series Fundamentals

In [13]:
# Creating Series with list and labels

labels = ["a", "b", "c"]
my_list: list[int] = [1, 2, 3]

my_series = pd.Series(data=my_list, index=labels)
my_series

,0
a,1
b,2
c,3


In [29]:
# creating series with dictionary

people: dict[str, int | str] = {"name": "Alice", "age": 25, "city": "NYC"}

people_series = pd.Series(people)
people_series


,0
name,Alice
age,25
city,NYC


In [15]:
# Creating DataFrames
df = pd.DataFrame(
    {
        "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
        "age": [25, 30, 35, 28, 22],
        "salary": [55000, 72000, 88000, 65000, 48000],
        "dept": ["Eng", "Marketing", "Eng", "HR", "Marketing"],
        "rating": [4.2, 3.8, 4.7, 4.0, 3.5],
    }
)

print(df)
print(f"\nShape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Dtypes:\n{df.dtypes}")

      name  age  salary       dept  rating
0    Alice   25   55000        Eng     4.2
1      Bob   30   72000  Marketing     3.8
2  Charlie   35   88000        Eng     4.7
3    Diana   28   65000         HR     4.0
4      Eve   22   48000  Marketing     3.5

Shape: (5, 5)
Columns: ['name', 'age', 'salary', 'dept', 'rating']
Dtypes:
name       object
age         int64
salary      int64
dept       object
rating    float64
dtype: object


In [16]:
# Series — a single column
salary_series = df["salary"]
print(type(salary_series))
print(salary_series.describe())

# Useful Series methods
print(f"\nMean   : {salary_series.mean():.0f}")
print(f"Median : {salary_series.median():.0f}")
print(f"Std    : {salary_series.std():.0f}")
print(f"Min    : {salary_series.min()}, Max: {salary_series.max()}")
print(
    f"Q1, Q3 : {salary_series.quantile(0.25):.0f}, {salary_series.quantile(0.75):.0f}"
)

<class 'pandas.core.series.Series'>
count        5.000000
mean     65600.000000
std      15533.834041
min      48000.000000
25%      55000.000000
50%      65000.000000
75%      72000.000000
max      88000.000000
Name: salary, dtype: float64

Mean   : 65600
Median : 65000
Std    : 15534
Min    : 48000, Max: 88000
Q1, Q3 : 55000, 72000


---
## 2. Indexing and Selection

In [17]:
# Column selection
print(df[["name", "salary"]])  # multiple columns → DataFrame
print()

# .loc — label-based (row label, column label)
print(df.loc[0, "salary"])  # row 0, column 'salary'
print(df.loc[1:3, ["name", "dept"]])  # rows 1-3, two columns

# .iloc — integer-position based (row index, col index)
print(df.iloc[0, 2])  # row 0, column 2 (salary)
print(df.iloc[:3, :3])  # first 3 rows, first 3 columns

      name  salary
0    Alice   55000
1      Bob   72000
2  Charlie   88000
3    Diana   65000
4      Eve   48000

55000
      name       dept
1      Bob  Marketing
2  Charlie        Eng
3    Diana         HR
55000
      name  age  salary
0    Alice   25   55000
1      Bob   30   72000
2  Charlie   35   88000


In [18]:
# Boolean filtering
high_earners = df[df["salary"] > 60000]
print("High earners (>60k):")
print(high_earners)

# Multiple conditions with & (and) and | (or)
eng_high = df[(df["dept"] == "Eng") & (df["salary"] > 60000)]
print("\nEngineers earning >60k:")
print(eng_high)

# .query() — readable string syntax
result = df.query("dept == 'Marketing' and rating >= 3.8")
print("\nMarketing with rating >= 3.8:")
print(result)

High earners (>60k):
      name  age  salary       dept  rating
1      Bob   30   72000  Marketing     3.8
2  Charlie   35   88000        Eng     4.7
3    Diana   28   65000         HR     4.0

Engineers earning >60k:
      name  age  salary dept  rating
2  Charlie   35   88000  Eng     4.7

Marketing with rating >= 3.8:
  name  age  salary       dept  rating
1  Bob   30   72000  Marketing     3.8


---
## 3. Adding, Modifying, and Deleting Columns

In [19]:
df2 = df.copy()

# Add new columns
df2["salary_k"] = df2["salary"] / 1000  # simple transform
df2["senior"] = df2["age"] >= 30  # boolean flag
df2["perf_category"] = pd.cut(
    df2["rating"],  # bin into categories
    bins=[0, 3.5, 4.0, 5.0],
    labels=["Low", "Mid", "High"],
)

print(df2[["name", "salary_k", "senior", "perf_category"]])

# Rename columns
df2 = df2.rename(columns={"name": "employee_name", "dept": "department"})
print("\nRenamed:", df2.columns.tolist())

# Drop columns
df2 = df2.drop(columns=["salary_k"])
print("After drop:", df2.columns.tolist())

      name  salary_k  senior perf_category
0    Alice      55.0   False          High
1      Bob      72.0    True           Mid
2  Charlie      88.0    True          High
3    Diana      65.0   False           Mid
4      Eve      48.0   False           Low

Renamed: ['employee_name', 'age', 'salary', 'department', 'rating', 'salary_k', 'senior', 'perf_category']
After drop: ['employee_name', 'age', 'salary', 'department', 'rating', 'senior', 'perf_category']


---
## 4. groupby — Aggregation by Category

This is one of the most used operations in DS. Think of it as SQL's `GROUP BY`.

In [20]:
# Basic groupby
dept_salary = df.groupby("dept")["salary"].agg(["mean", "median", "std", "count"])
print("Salary statistics by department:")
print(dept_salary.round(0))

# Multiple aggregations on multiple columns
dept_stats = df.groupby("dept").agg(
    avg_salary=("salary", "mean"),
    avg_rating=("rating", "mean"),
    headcount=("name", "count"),
    avg_age=("age", "mean"),
)
print("\nDepartment summary:")
print(dept_stats.round(2))

Salary statistics by department:
              mean   median      std  count
dept                                       
Eng        71500.0  71500.0  23335.0      2
HR         65000.0  65000.0      NaN      1
Marketing  60000.0  60000.0  16971.0      2

Department summary:
           avg_salary  avg_rating  headcount  avg_age
dept                                                 
Eng           71500.0        4.45          2     30.0
HR            65000.0        4.00          1     28.0
Marketing     60000.0        3.65          2     26.0


In [21]:
# transform: apply group function but keep original shape (for normalizing within groups)
df["salary_rank_in_dept"] = df.groupby("dept")["salary"].rank(pct=True)
df["salary_zscore_dept"] = df.groupby("dept")["salary"].transform(
    lambda x: (x - x.mean()) / x.std(ddof=0)
)
print(
    df[["name", "dept", "salary", "salary_rank_in_dept", "salary_zscore_dept"]].round(3)
)

      name       dept  salary  salary_rank_in_dept  salary_zscore_dept
0    Alice        Eng   55000                  0.5                -1.0
1      Bob  Marketing   72000                  1.0                 1.0
2  Charlie        Eng   88000                  1.0                 1.0
3    Diana         HR   65000                  1.0                 NaN
4      Eve  Marketing   48000                  0.5                -1.0


---
## 5. Merging DataFrames

Like SQL JOINs. Essential for combining multiple data sources.

In [22]:
employees = pd.DataFrame(
    {
        "emp_id": [1, 2, 3, 4, 5],
        "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
        "dept_id": [10, 20, 10, 30, 20],
    }
)

departments = pd.DataFrame(
    {
        "dept_id": [10, 20, 30, 40],
        "dept_name": ["Engineering", "Marketing", "HR", "Finance"],
        "location": ["NYC", "LA", "Chicago", "NYC"],
    }
)

# Inner join — only matching rows from both
inner = pd.merge(employees, departments, on="dept_id", how="inner")
print("INNER JOIN:")
print(inner)

# Left join — all employees, even if dept not found
emp_with_unmatched = pd.DataFrame(
    {
        "emp_id": [1, 2, 3, 6],
        "name": ["Alice", "Bob", "Charlie", "Frank"],
        "dept_id": [10, 20, 10, 99],
    }
)
left = pd.merge(emp_with_unmatched, departments, on="dept_id", how="left")
print("\nLEFT JOIN (Frank has no matching dept):")
print(left)

INNER JOIN:
   emp_id     name  dept_id    dept_name location
0       1    Alice       10  Engineering      NYC
1       2      Bob       20    Marketing       LA
2       3  Charlie       10  Engineering      NYC
3       4    Diana       30           HR  Chicago
4       5      Eve       20    Marketing       LA

LEFT JOIN (Frank has no matching dept):
   emp_id     name  dept_id    dept_name location
0       1    Alice       10  Engineering      NYC
1       2      Bob       20    Marketing       LA
2       3  Charlie       10  Engineering      NYC
3       6    Frank       99          NaN      NaN


---
## 6. Missing Data Handling

Real datasets always have missing values. You must understand why data is missing before deciding how to handle it:
- **MCAR** (Missing Completely At Random): random data entry omissions
- **MAR** (Missing At Random): missingness depends on other observed variables
- **MNAR** (Missing Not At Random): missingness depends on the missing value itself (e.g., very high income not reported)

In [23]:
# Create a dataset with missing values
n = 200
df_missing = pd.DataFrame(
    {
        "age": np.where(np.random.rand(n) < 0.05, np.nan, np.random.randint(20, 65, n)),
        "income": np.where(
            np.random.rand(n) < 0.15, np.nan, np.random.normal(60000, 20000, n)
        ),
        "score": np.where(
            np.random.rand(n) < 0.10, np.nan, np.random.uniform(0, 100, n)
        ),
        "city": np.where(
            np.random.rand(n) < 0.08,
            None,
            np.random.choice(["NYC", "LA", "Chicago"], n),
        ),
    }
)

# Summary of missing values
print("Missing value summary:")
missing = pd.DataFrame(
    {
        "count": df_missing.isnull().sum(),
        "percent": (df_missing.isnull().sum() / len(df_missing) * 100).round(1),
    }
)
print(missing)

Missing value summary:
        count  percent
age        11      5.5
income     36     18.0
score      23     11.5
city       18      9.0


In [24]:
# Strategies for handling missing values
df_clean = df_missing.copy()

# 1. Drop rows with missing values (only for small % missing)
df_dropped = df_clean.dropna()  # drops any row with ANY null
df_dropped_subset = df_clean.dropna(subset=["age"])  # only drop where age is null
print(f"Original: {len(df_clean)} rows, after dropna: {len(df_dropped)} rows")

# 2. Fill with mean/median (for numerical, skewed → use median)
df_clean["age"].fillna(df_clean["age"].median(), inplace=True)
df_clean["income"].fillna(df_clean["income"].median(), inplace=True)
df_clean["score"].fillna(df_clean["score"].mean(), inplace=True)

# 3. Fill with mode (for categorical)
df_clean["city"].fillna(df_clean["city"].mode()[0], inplace=True)

print(f"\nMissing after fill: {df_clean.isnull().sum().sum()}")

# 4. Forward fill / backward fill (for time series)
ts = pd.Series([1, np.nan, np.nan, 4, 5, np.nan, 7])
print(f"\nOriginal:  {ts.tolist()}")
print(f"ffill:     {ts.ffill().tolist()}")
print(f"bfill:     {ts.bfill().tolist()}")
print(f"interpolate: {ts.interpolate().tolist()}")

Original: 200 rows, after dropna: 124 rows

Missing after fill: 0

Original:  [1.0, nan, nan, 4.0, 5.0, nan, 7.0]
ffill:     [1.0, 1.0, 1.0, 4.0, 5.0, 5.0, 7.0]
bfill:     [1.0, 4.0, 4.0, 4.0, 5.0, 7.0, 7.0]
interpolate: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]


/tmp/ipykernel_518/2572711839.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean["age"].fillna(df_clean["age"].median(), inplace=True)
/tmp/ipykernel_518/2572711839.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

---
## 7. apply and Vectorized Operations

In [25]:
df_apply = df.copy()

# apply on a Series — row by row transformation
df_apply["salary_band"] = df_apply["salary"].apply(
    lambda x: "Low" if x < 60000 else ("Mid" if x < 80000 else "High")
)


# apply on DataFrame axis=1 — row-wise custom logic
def performance_label(row):
    if row["rating"] >= 4.5:
        return "Star"
    elif row["rating"] >= 4.0 and row["salary"] > 60000:
        return "Solid"
    else:
        return "Developing"


df_apply["label"] = df_apply.apply(performance_label, axis=1)

print(df_apply[["name", "salary", "rating", "salary_band", "label"]])

# NOTE: prefer vectorized operations over apply for performance
# Vectorized (fast):
df_apply["salary_normalized"] = (
    df_apply["salary"] - df_apply["salary"].mean()
) / df_apply["salary"].std()
print("\nZ-score normalized salary:", df_apply["salary_normalized"].round(3).tolist())

      name  salary  rating salary_band       label
0    Alice   55000     4.2         Low  Developing
1      Bob   72000     3.8         Mid  Developing
2  Charlie   88000     4.7        High        Star
3    Diana   65000     4.0         Mid       Solid
4      Eve   48000     3.5         Low  Developing

Z-score normalized salary: [-0.682, 0.412, 1.442, -0.039, -1.133]


---
## 8. Reshaping — pivot_table, melt, stack/unstack

In [26]:
# Create a wider dataset
sales = pd.DataFrame(
    {
        "month": ["Jan", "Jan", "Feb", "Feb", "Mar", "Mar"],
        "product": ["A", "B", "A", "B", "A", "B"],
        "region": ["North", "North", "South", "South", "North", "North"],
        "units": [120, 85, 95, 110, 135, 90],
        "revenue": [12000, 9000, 9500, 11000, 13500, 9500],
    }
)

# pivot_table — like an Excel pivot
pivot = pd.pivot_table(
    sales,
    values="revenue",
    index="month",
    columns="product",
    aggfunc="sum",
    margins=True,  # add totals row/column
)
print("Pivot Table — Revenue by Month and Product:")
print(pivot)

# melt — wide to long format (tidy data)
wide_df = pd.DataFrame(
    {
        "employee": ["Alice", "Bob"],
        "q1_sales": [100, 120],
        "q2_sales": [130, 115],
        "q3_sales": [125, 140],
    }
)
long_df = wide_df.melt(id_vars="employee", var_name="quarter", value_name="sales")
print("\nMelted (wide → long):")
print(long_df)

Pivot Table — Revenue by Month and Product:
product      A      B    All
month                       
Feb       9500  11000  20500
Jan      12000   9000  21000
Mar      13500   9500  23000
All      35000  29500  64500

Melted (wide → long):
  employee   quarter  sales
0    Alice  q1_sales    100
1      Bob  q1_sales    120
2    Alice  q2_sales    130
3      Bob  q2_sales    115
4    Alice  q3_sales    125
5      Bob  q3_sales    140


---
## Summary — Pandas Operations

| Operation | Method |
|-----------|--------|
| Select columns | `df[col]`, `df[[c1, c2]]` |
| Label indexing | `df.loc[rows, cols]` |
| Position indexing | `df.iloc[rows, cols]` |
| Filter rows | `df[condition]`, `df.query()` |
| Group aggregation | `df.groupby(col).agg(...)` |
| Group transform | `df.groupby(col).transform(...)` |
| Join tables | `pd.merge(df1, df2, on, how)` |
| Missing summary | `df.isnull().sum()` |
| Fill missing | `df.fillna(value)` |
| Drop missing | `df.dropna(subset)` |
| Row-wise apply | `df.apply(func, axis=1)` |
| Pivot | `pd.pivot_table(df, values, index, columns)` |
| Unpivot | `df.melt(id_vars, var_name, value_name)` |